# 02 - Validacao de Qualidade dos Dados

## Objetivo
Validar qualidade dos dados: valores ausentes, duplicatas e outliers.

## Fluxo
1. Carregar dados
2. Verificar duplicatas
3. Detectar valores ausentes
4. Identificar outliers (metodo IQR)
5. Relatorio de qualidade

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Carregar dados

In [ ]:
caminho_dados = Path('../dados/brutos/indicadores_consolidados.csv')
df = pd.read_csv(caminho_dados)
df['data'] = pd.to_datetime(df['data'])

print(f'Dados carregados: {df.shape[0]} linhas x {df.shape[1]} colunas')

## 2. Verificar duplicatas

In [ ]:
# Duplicatas totais
duplicatas_total = df.duplicated().sum()

# Duplicatas por coluna data (principal problema)
duplicatas_data = df.duplicated(subset=['data'], keep=False).sum()

print('VERIFICACAO DE DUPLICATAS:')
print(f'Linhas duplicadas (todas as colunas): {duplicatas_total}')
print(f'Linhas com data duplicada: {duplicatas_data}')

if duplicatas_total == 0:
    print('Status: OK - Sem duplicatas!')

## 3. Verificar valores ausentes

In [ ]:
# Contar valores ausentes
ausentes = df.isnull().sum()
ausentes_pct = (df.isnull().sum() / len(df) * 100).round(2)

print('VERIFICACAO DE VALORES AUSENTES:')
print('\nColuna | Quantidade | Percentual')
print('-' * 40)
for col in ausentes.index:
    print(f'{col:30} | {ausentes[col]:10} | {ausentes_pct[col]:6.2f}%')

total_ausentes = ausentes.sum()
if total_ausentes == 0:
    print('
Status: OK - Sem valores ausentes!')

## 4. Detectar outliers (Metodo IQR)

In [ ]:
def detectar_outliers_iqr(df, coluna):
    Q1 = df[coluna].quantile(0.25)
    Q3 = df[coluna].quantile(0.75)
    IQR = Q3 - Q1
    limite_inf = Q1 - 1.5 * IQR
    limite_sup = Q3 + 1.5 * IQR
    outliers = df[(df[coluna] < limite_inf) | (df[coluna] > limite_sup)]
    return outliers, limite_inf, limite_sup

# Colunas numericas
colunas_numericas = df.select_dtypes(include=[np.number]).columns

print('DETECCAO DE OUTLIERS (METODO IQR):')
print('=' * 70)

outliers_resumo = {}
for col in colunas_numericas:
    outliers, lim_inf, lim_sup = detectar_outliers_iqr(df, col)
    outliers_resumo[col] = len(outliers)
    pct = len(outliers) / len(df) * 100
    print(f'{col}:')
    print(f'  Quantidade: {len(outliers):3} ({pct:5.2f}%)')
    print(f'  Intervalo valido: [{lim_inf:8.2f}, {lim_sup:8.2f}]')
    if len(outliers) > 0:
        print(f'  Valores outliers: {outliers[col].values.round(2)}')
    print()

## 5. Visualizar outliers com boxplot

In [ ]:
# Criar boxplots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Deteccao de Outliers - Boxplot', fontsize=16, fontweight='bold')

axes_flat = axes.flatten()
for i, col in enumerate(colunas_numericas):
    sns.boxplot(y=df[col], ax=axes_flat[i], palette='Set2')
    axes_flat[i].set_title(f'{col}')
    axes_flat[i].set_ylabel('Valor')

plt.tight_layout()
plt.show()

## 6. Relatorio de qualidade

In [ ]:
print('\n' + '=' * 70)
print('RELATORIO FINAL DE QUALIDADE')
print('=' * 70)

status_duplicatas = 'PASSOU' if duplicatas_total == 0 else 'FALHOU'
status_ausentes = 'PASSOU' if total_ausentes == 0 else 'FALHOU'
status_outliers = 'PASSOU' if sum(outliers_resumo.values()) == 0 else 'AVISO'

print(f'Duplicatas: {status_duplicatas} ({duplicatas_total} encontradas)')
print(f'Valores ausentes: {status_ausentes} ({total_ausentes} encontrados)')
print(f'Outliers: {status_outliers} ({sum(outliers_resumo.values())} encontrados)')

status_geral = 'OK' if status_duplicatas == 'PASSOU' and status_ausentes == 'PASSOU' else 'COM PROBLEMAS'
print(f'
Status Geral: {status_geral}')
print('=' * 70)

## 7. Limpeza basica (se necessario)

In [ ]:
# Salvar dados validados
caminho_processado = Path('../dados/processados/dados_validados.csv')
df.to_csv(caminho_processado, index=False, encoding='utf-8')

print(f'Dados validados salvos em: {caminho_processado}')